In [ ]:
from pyspark.sql import SparkSession, functions as F
from IPython.display import display

spark = (SparkSession.builder
    .appName("olist-03-exploration")
    .config("spark.sql.warehouse.dir", "hdfs:///user/hive/warehouse")
    .enableHiveSupport()
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

RAW    = "hdfs:///data/olist/raw"
BUCKET = "gs://olist-sujinan-2026"
DB     = "olist"

def show(df, n=20):
    display(df.limit(n).toPandas())

def pdf(df, n=1000):
    return df.limit(n).toPandas()

In [ ]:
names = ["customers", "orders", "order_items", "payments", "reviews",
         "products", "sellers", "geolocation", "categories"]

dfs = {n: spark.table(f"{DB}.bronze_{n}") for n in names}
print(len(dfs), "bronze tables loaded")

# Remove rows missing the key field

In [ ]:
orders_c = dfs["orders"].na.drop(
    subset=["order_id", "customer_id", "order_status"]
)

In [ ]:
orders_c = (orders_c
    .withColumn("is_delivery_date_missing",
                F.col("order_delivered_customer_date").isNull())

    .withColumn("order_delivered_customer_date",
                F.coalesce(F.col("order_delivered_customer_date"),
                           F.lit("9999-12-31").cast("timestamp")))
)

# Fill in numerical values

In [ ]:
from pyspark.ml.feature import Imputer

imputer = Imputer(
    inputCols=["payment_value"],
    outputCols=["payment_value_imputed"],
).setStrategy("median")

payments_c = imputer.fit(dfs["payments"]).transform(dfs["payments"])

# Fix data types that are semantically incorrect

In [ ]:
customers_c = (dfs["customers"]
    .withColumn("customer_zip_code_prefix",
                F.col("customer_zip_code_prefix").cast("string"))
    .dropDuplicates(["customer_id"])
)

# Reduce geolocation to one line per zip code.

In [ ]:
geo_c = (dfs["geolocation"]
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.avg("geolocation_lat").alias("geo_lat"),
        F.avg("geolocation_lng").alias("geo_lng"),
        F.first("geolocation_city").alias("geo_city"),
        F.first("geolocation_state").alias("geo_state"),
    )
    .withColumn("customer_zip_code_prefix",
                F.col("geolocation_zip_code_prefix").cast("string"))
    .drop("geolocation_zip_code_prefix")
)

print("geolocation:", dfs["geolocation"].count(), "->", geo_c.count())

# Normalize category values

In [ ]:
payments_c = payments_c.withColumn(
    "payment_method",
    F.when(F.col("payment_type") == "credit_card", "Credit Card")
     .when(F.col("payment_type") == "debit_card",  "Debit Card")
     .when(F.col("payment_type") == "boleto",      "Bank Slip")
     .when(F.col("payment_type") == "voucher",     "Voucher")
     .otherwise("Other")
)

# Remove outliers using quantiles.

In [ ]:
low, high = dfs["order_items"].approxQuantile("price", [0.01, 0.99], 0.001)
print(f"ช่วงราคา {low:.2f} – {high:.2f}")

items_c = dfs["order_items"].filter(
    (F.col("price") >= low) & (F.col("price") <= high)
)
print(f"{dfs['order_items'].count():,} → {items_c.count():,}")

# Feature engineering - Group product sizes + Translate category names

In [ ]:
products_c = (dfs["products"]
    .withColumn(
        "product_size_category",
        F.when(F.col("product_weight_g") < 500,  "small")
         .when(F.col("product_weight_g") < 2000, "medium")
         .otherwise("large"))
    .join(F.broadcast(dfs["categories"]), "product_category_name", "left")
)

In [ ]:
need = ["orders_c", "payments_c", "customers_c", "geo_c", "items_c", "products_c"]
for v in need:
    print(f"{v:<14s} {'OK' if v in globals() else 'ยังไม่ได้สร้าง'}")

# Save as silver tables.

In [ ]:
silver = {
    "customers":   customers_c,
    "orders":      orders_c,
    "order_items": items_c,
    "payments":    payments_c,
    "products":    products_c,
    "sellers":     dfs["sellers"],
    "reviews":     dfs["reviews"],
    "geolocation": geo_c,
}

for name, df in silver.items():
    (df.write.mode("overwrite").format("parquet")
       .saveAsTable(f"{DB}.silver_{name}"))
    print(f"saved silver_{name}: {df.count():,} rows")

for name, df in silver.items():
    df.write.mode("overwrite").parquet(f"{BUCKET}/silver/{name}")

In [ ]:
for name in silver:
    n = spark.table(f"{DB}.silver_{name}").count()
    print(f"silver_{name:<14s} {n:>9,} rows")

# Fix excess reviews, inserting a line break inside quotation marks.